
# 📘 NOTEBOOK : TaaSim — Remapping Porto → Casablanca (Version Finale)

**CELLULE 0 — Installations**

In [1]:
!pip install -r /home/jovyan/requirements.txt -q
!pip install geopandas shapely folium osmnx networkx -q

**CELLULE 1 — Initialisation Spark**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import folium
import json

spark = SparkSession.builder \
    .appName("TaaSim-Remapping") \
    .config("spark.driver.memory", "4g") \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "taasim") \
    .config("spark.hadoop.fs.s3a.secret.key", "taasim123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print("✅ Spark prêt avec MinIO")

✅ Spark prêt avec MinIO


**🗺 CELLULE 2 — Chargement GeoJSON Casablanca**

In [3]:
import geopandas as gpd
from shapely.geometry import Point

GEOJSON = "/home/jovyan/work/data/Arrondissements.geojson"

gdf_arr = gpd.read_file(GEOJSON)
CASA_POLYGON = gdf_arr.unary_union
CASA_LON_MIN, CASA_LAT_MIN, CASA_LON_MAX, CASA_LAT_MAX = gdf_arr.total_bounds

print("GeoJSON chargé ✔")
print("Casablanca BBOX :", CASA_LON_MIN, CASA_LAT_MIN, CASA_LON_MAX, CASA_LAT_MAX)

GeoJSON chargé ✔
Casablanca BBOX : -7.746445 33.493772399999976 -7.4574165 33.64087470000003


/tmp/ipykernel_30351/82506348.py:7: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  CASA_POLYGON = gdf_arr.unary_union


**🧭 CELLULE 3 — BBOX Porto (calcul automatique)**

In [4]:
import json

df_porto = spark.read.csv("s3a://raw/porto-trips/train.csv", header=True)

def extract_bounds(pdf):
    lon_min = lon_max = lat_min = lat_max = None
    for row in pdf:
        if not row or row == "[]":
            continue
        pts = json.loads(row)
        for lon, lat in pts:
            lon_min = lon if lon_min is None else min(lon_min, lon)
            lon_max = lon if lon_max is None else max(lon_max, lon)
            lat_min = lat if lat_min is None else min(lat_min, lat)
            lat_max = lat if lat_max is None else max(lat_max, lat)
    return lon_min, lon_max, lat_min, lat_max

pdf_poly = df_porto.select("POLYLINE").limit(50000).toPandas()["POLYLINE"]
PORTO_LON_MIN, PORTO_LON_MAX, PORTO_LAT_MIN, PORTO_LAT_MAX = extract_bounds(pdf_poly)

print("BBOX Porto :")
PORTO_LON_MIN, PORTO_LON_MAX, PORTO_LAT_MIN, PORTO_LAT_MAX

BBOX Porto :


(-12.154527, -6.958539, 38.664981, 45.657225)

**🔄 CELLULE 4 — Fonctions de remapping Porto → Casablanca**

In [5]:
def linear_map(value, src_min, src_max, dst_min, dst_max):
    ratio = (value - src_min) / (src_max - src_min)
    return dst_min + ratio * (dst_max - dst_min)

def porto_to_casa_linear(lon, lat):
    lon_c = linear_map(lon, PORTO_LON_MIN, PORTO_LON_MAX,
                       CASA_LON_MIN, CASA_LON_MAX)
    lat_c = linear_map(lat, PORTO_LAT_MIN, PORTO_LAT_MAX,
                       CASA_LAT_MIN, CASA_LAT_MAX)
    return lon_c, lat_c

def project_inside_casa_polygon(lon, lat):
    p = Point(lon, lat)
    if CASA_POLYGON.contains(p):
        return lon, lat
    b = CASA_POLYGON.boundary
    near = b.interpolate(b.project(p))
    return near.x, near.y

def porto_to_casa_complete(lon, lat):
    lon_c, lat_c = porto_to_casa_linear(lon, lat)
    return project_inside_casa_polygon(lon_c, lat_c)

**🔁 CELLULE 5 — UDF Spark : transformation du polyline**

In [6]:
def remap_polyline(polyline_str):
    try:
        pts = json.loads(polyline_str)
        remapped = []
        for lon, lat in pts:
            lon2, lat2 = porto_to_casa_complete(lon, lat)
            remapped.append([round(lon2,6), round(lat2,6)])
        return json.dumps(remapped)
    except:
        return None

remap_udf = F.udf(remap_polyline, StringType())

**📥 CELLULE 6 — Application au dataset complet**

In [7]:
df_remapped = df_porto.withColumn(
    "remapped_polyline",
    remap_udf(F.col("POLYLINE"))
)

df_remapped.show(3, truncate=False)

+-------------------+---------+-----------+------------+--------+----------+--------+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

**🧪 CELLULE 7 — Validation rapide (3 points)**

In [8]:
import pandas as pd

def check_fast(polyline_str):
    pts = json.loads(polyline_str)
    if len(pts) < 3:
        return False
    test_pts = [pts[0], pts[len(pts)//2], pts[-1]]
    return all(CASA_POLYGON.contains(Point(lon,lat)) for lon,lat in test_pts)

pdf = df_remapped.select("remapped_polyline").limit(5000).toPandas()
pdf["inside"] = pdf["remapped_polyline"].apply(check_fast)

pdf["inside"].value_counts()

inside
True     4861
False     139
Name: count, dtype: int64

**🗺 CELLULE 8 — Visualisation Folium avant/après**

In [9]:
import folium

def plot_before_after(porto_polyline_str, casa_polyline_str):
    pts_porto = json.loads(porto_polyline_str)
    pts_casa  = json.loads(casa_polyline_str)

    m = folium.Map(location=[33.57, -7.59], zoom_start=12)

    folium.PolyLine([(lat,lon) for lon,lat in pts_porto],
                    color="blue", opacity=0.6, weight=3).add_to(m)

    folium.PolyLine([(lat,lon) for lon,lat in pts_casa],
                    color="red", opacity=0.9, weight=4).add_to(m)

    return m

In [10]:
import folium
import json

def plot_multiple_trips(rows):
    # Carte centrée sur Casablanca
    m = folium.Map(location=[33.5731, -7.5898], zoom_start=12)

    colors = ["red", "blue", "green", "purple", "orange",
              "black", "darkred", "cadetblue", "pink", "gray"]

    for i, row in enumerate(rows):
        try:
            porto = json.loads(row["POLYLINE"])
            casa  = json.loads(row["remapped_polyline"])

            # Porto (avant)
            folium.PolyLine(
                locations=[(lat, lon) for lon, lat in porto],
                color="blue",
                weight=2,
                opacity=0.4
            ).add_to(m)

            # Casa (après)
            folium.PolyLine(
                locations=[(lat, lon) for lon, lat in casa],
                color=colors[i % len(colors)],
                weight=4,
                opacity=0.8
            ).add_to(m)

        except:
            continue

    return m

In [11]:
import folium

def add_casa_boundary(m, casa_polygon, color="black"):
    """
    Ajoute la frontière de Casablanca à une carte Folium existante
    """

    # Si MultiPolygon → plusieurs morceaux
    if casa_polygon.geom_type == "MultiPolygon":
        polygons = casa_polygon.geoms
    else:
        polygons = [casa_polygon]

    for poly in polygons:
        coords = list(poly.exterior.coords)

        folium.PolyLine(
            locations=[(lat, lon) for lon, lat in coords],
            color=color,
            weight=3,
            opacity=0.9
        ).add_to(m)

    return m

**🎨 CELLULE 9 — Afficher un échantillon de trajets**

In [12]:
df_small = df_remapped.limit(1000).cache()
df_small.count()


1000

In [13]:
rows = df_small.sample(0.01).limit(10).collect()

m = plot_multiple_trips(rows)

m.save("/home/jovyan/work/data/multi_trips.html")
m = add_casa_boundary(m, CASA_POLYGON)
print("Visualisation sauvegardée dans visualization.html")
m


Visualisation sauvegardée dans visualization.html


**🏙 CELLULE 10 — Construction du zone_mapping.csv**

In [14]:
# %% [markdown]
# CELLULE 2.5 — Création du mapping zone_id depuis le GeoJSON

# %%
BASE_PATH="/home/jovyan/work"
def create_zone_mapping_from_geojson(gdf):
    """
    Crée un mapping entre zone_id et les arrondissements du GeoJSON
    """
    zone_mapping = []
    
    for idx, row in gdf.iterrows():
        # Extraire les informations
        arr_name = row.get("Arrondissement", f"Arrondissement_{idx}")
        prefecture = row.get("Prefecture", "N/A")
        population = row.get("Population", 0)
        superficie = row.get("Superficie", 0)
        
        # Obtenir le bounding box de l'arrondissement
        bounds = row.geometry.bounds
        lon_min, lat_min, lon_max, lat_max = bounds
        
        # Déterminer le type de zone et le tarif
        zone_type = "residential"  # valeur par défaut
        commercial_keywords = ["Maarif", "Anfa", "Belyout", "Mers Sultan", "El Fida"]
        industrial_keywords = ["Ain Sebaa", "Hay Mohammadi", "Bernoussi"]
        
        for kw in commercial_keywords:
            if kw.lower() in arr_name.lower():
                zone_type = "commercial"
                break
        for kw in industrial_keywords:
            if kw.lower() in arr_name.lower():
                zone_type = "industrial"
                break
        
        # Tarif de base (MAD)
        expensive_zones = ["Anfa", "Maarif", "Belyout", "Mers Sultan", "Ain Diab"]
        base_fare = 8.0
        for zone in expensive_zones:
            if zone.lower() in arr_name.lower():
                base_fare = 10.0
                break
        if population > 300000:
            base_fare = 7.0
        
        # Centre de l'arrondissement
        centroid = row.geometry.centroid
        centroid_lon = centroid.x
        centroid_lat = centroid.y
        
        zone_mapping.append({
            "zone_id": idx + 1,
            "zone_name": arr_name,
            "prefecture": prefecture,
            "zone_type": zone_type,
            "population": population,
            "superficie_m2": superficie,
            "centroid_lon": centroid_lon,
            "centroid_lat": centroid_lat,
            "bbox_lon_min": lon_min,
            "bbox_lon_max": lon_max,
            "bbox_lat_min": lat_min,
            "bbox_lat_max": lat_max,
            "base_fare_mad": base_fare,
        })
    
    return zone_mapping



**💾 CELLULE 11 — Export final**

In [16]:
# Créer le mapping à partir du GeoJSON si disponible

if gdf_arr is not None:
    zone_mapping_geo = create_zone_mapping_from_geojson(gdf_arr)
    
    # Afficher le mapping créé
    print("✅ Mapping créé à partir du GeoJSON :")
    for zone in zone_mapping_geo[:5]:
        print(f"   {zone['zone_id']}: {zone['zone_name']} ({zone['zone_type']}) - {zone['base_fare_mad']} MAD")
    
    # Sauvegarder en CSV pour référence
    import pandas as pd
    pd.DataFrame(zone_mapping_geo).to_csv(f"{BASE_PATH}/data/zone_mapping_geojson.csv", index=False)
    print(f"✅ Sauvegardé dans {BASE_PATH}/data/zone_mapping_geojson.csv")
else:
    zone_mapping_geo = None
    print("⚠️ GeoJSON non disponible, utilisation du mapping manuel")

✅ Mapping créé à partir du GeoJSON :
   1: Mechouar Casablanca (residential) - 8.0 MAD
   2: arrondissement d'Anfa (أنفا (المقاطعة (commercial) - 10.0 MAD
   3: arrondissement de Hay Hassani (الحي الحسني (المقاطعة (residential) - 7.0 MAD
   4: arrondissement de Sidi Moumen (سيدي مومن (المقاطعة (residential) - 7.0 MAD
   5: arrondissement de Moulay Rachid (مولاي رشيد (المقاطعة (residential) - 8.0 MAD
✅ Sauvegardé dans /home/jovyan/work/data/zone_mapping_geojson.csv


In [17]:
import osmnx as ox

G = ox.graph_from_place("Casablanca, Morocco", network_type="drive")
print("✅ Graphe routier chargé")

✅ Graphe routier chargé


In [ ]:
import networkx as nx

def snap_to_road(polyline_str):
    try:
        pts = json.loads(polyline_str)

        # début et fin
        start_lon, start_lat = pts[0]
        end_lon, end_lat     = pts[-1]

        # trouver les noeuds les plus proches
        start_node = ox.distance.nearest_nodes(G, start_lon, start_lat)
        end_node   = ox.distance.nearest_nodes(G, end_lon, end_lat)

        # plus court chemin
        route = nx.shortest_path(G, start_node, end_node, weight="length")

        # convertir en coords GPS
        route_coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in route]

        return route_coords

    except:
        return None

In [ ]:
row = rows[0]

real_route = snap_to_road(row["remapped_polyline"])

m = folium.Map(location=real_route[0], zoom_start=13)
m = add_casa_boundary(m, CASA_POLYGON)
folium.PolyLine(real_route, color="green", weight=5).add_to(m)


m

In [ ]:
import folium
import json

def get_trip_coordinates(row):
    """
    Extrait les coordonnées du trajet Casablanca à partir d'une ligne.
    Retourne une liste de points [[lat, lon], ...]
    """
    try:
        casa_points = json.loads(row["remapped_polyline"])
        if len(casa_points) >= 2:
            return [(lat, lon) for lon, lat in casa_points]
        else:
            return None
    except Exception as e:
        print(f"Erreur : {e}")
        return None

# ============================================
# Utilisation : afficher les 2 trajets sur la même carte
# ============================================

# 1. Récupérer la ligne
row = rows[0]

# 2. Obtenir les coordonnées des deux trajets
fake_route = get_trip_coordinates(row)           # Trajet normal (sans snap)
real_route = snap_to_road(row["remapped_polyline"])  # Trajet avec snap

# 3. Vérifier que les deux existent
if real_route is None:
    print("❌ real_route est vide")
if fake_route is None:
    print("❌ fake_route est vide")

# 4. Créer une SEULE carte
if real_route and len(real_route) >= 2:
    m = folium.Map(location=real_route[0], zoom_start=13)
    
    # Ajouter le trajet real_route (vert)
    folium.PolyLine(
        locations=real_route,
        color="green",
        weight=5,
        opacity=0.8,
        tooltip="Trajet avec snap_to_road"
    ).add_to(m)
    
    # Ajouter le trajet fake_route (rouge) s'il existe
    if fake_route and len(fake_route) >= 2:
        folium.PolyLine(
            locations=fake_route,
            color="red",
            weight=3,
            opacity=0.6,
            tooltip="Trajet normal (sans snap)"
        ).add_to(m)
    
    # Ajouter les points de départ et arrivée
    folium.CircleMarker(
        location=real_route[0],
        radius=6,
        color="green",
        fill=True,
        popup="Départ"
    ).add_to(m)
    
    folium.CircleMarker(
        location=real_route[-1],
        radius=6,
        color="red",
        fill=True,
        popup="Arrivée"
    ).add_to(m)
    
    # Afficher la carte
    m
else:
    print("❌ Impossible d'afficher la carte")
    
def compare_trips_on_map(row):
    """
    Compare sur une même carte :
    - Trajet normal (rouge)
    - Trajet avec snap_to_road (vert)
    """
    # Extraire les coordonnées
    fake_route = get_trip_coordinates(row)
    real_route = snap_to_road(row["remapped_polyline"])
    
    if not real_route or len(real_route) < 2:
        print("❌ Trajet invalide")
        return None
    
    # Créer la carte
    m = folium.Map(location=real_route[0], zoom_start=13)
    
    # Trajet sans snap (rouge)
    if fake_route and len(fake_route) >= 2:
        folium.PolyLine(
            locations=fake_route,
            color="red",
            weight=3,
            opacity=0.5,
            tooltip="Sans snap_to_road"
        ).add_to(m)
    
    # Trajet avec snap (vert)
    folium.PolyLine(
        locations=real_route,
        color="green",
        weight=5,
        opacity=0.8,
        tooltip="Avec snap_to_road"
    ).add_to(m)
    
    # Points de départ et arrivée
    folium.CircleMarker(real_route[0], radius=6, color="green", fill=True, popup="Départ").add_to(m)
    folium.CircleMarker(real_route[-1], radius=6, color="red", fill=True, popup="Arrivée").add_to(m)
    
    # Ajouter une légende
    legend_html = """
    <div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
         background: white; padding: 10px; border: 1px solid #ccc;
         border-radius: 5px; font-size: 12px;">
        <b>Légende</b><br>
        <span style="color:green">●</span> Trajet avec snap_to_road<br>
        <span style="color:red">●</span> Trajet sans snap<br>
        <span style="color:green">●</span> Départ<br>
        <span style="color:red">●</span> Arrivée
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    
    return m

# Utilisation
m = compare_trips_on_map(row)
m